# Notebook 05 — Model Training

**Input:** outputs dari nb01 (splits), nb03 (mol representations), nb04 (protein embeddings)  
**GPU:** strongly recommended (RTX 3060 or better)

| Section | Description |
|---------|-------------|
| 1 | Model architecture (encoders → cross-attention → MoE → heads) |
| 2 | Data loading & Dataset/DataLoader |
| 3 | Optuna hyperparameter search (10-fold scaffold CV) |
| 4 | Final retrain on full train pool with best config |
| 5 | Save checkpoint |

> **DEMO mode:** set `demo.enabled: true` → 1-fold CV, 5 epochs, d=64. Runs in ~10 min on CPU.

---
## 0 — Imports & Config

In [ ]:
import sys, os, gc, json, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.nn import radius_graph
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import optuna
from optuna.pruners import MedianPruner
from optuna.trial import TrialState

warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

with open(REPO_ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

SEED      = CFG['seed']
DEMO_MODE = CFG['demo']['enabled']
torch.manual_seed(SEED)
np.random.seed(SEED)

SPLIT_ROOT   = Path(CFG['paths']['split_root'])
MOL_REP_ROOT = Path(CFG['paths']['mol_rep_root'])
TRAIN_ROOT   = Path(CFG['paths']['training_root'])
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

PROT_EMB_PATH   = MOL_REP_ROOT / 'protein_embeddings' / 'all_proteins_esm2_mean.pt'
MORD_PREPROC_PATH = MOL_REP_ROOT / 'mordred_preproc.pkl'

TASKS    = [t['name'] for t in CFG['targets']]
N_TASKS  = len(TASKS)
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

CHEMBERTA_MODEL   = 'seyonec/ChemBERTa-zinc-base-v1'
CHEMBERTA_DIM     = 768
CHEMBERTA_MAX_LEN = 512
ESM2_DIM          = 1280

# Demo vs full settings
if DEMO_MODE:
    N_OPTUNA_TRIALS = 2
    OPTUNA_FOLDS    = 1
    OPTUNA_EPOCHS   = 5
    FINAL_EPOCHS    = 5
    DEMO_CFG = {'latent_dim':64,'mlp_hidden':64,'mlp_layers':2,'gat_layers':2,
                'gat_heads':2,'schnet_hidden':32,'schnet_interactions':1,
                'attn_heads':2,'dropout':0.2,'lr':1e-4,'weight_decay':1e-4,
                'batch_size':8,'unfreeze_epoch':999}
else:
    N_OPTUNA_TRIALS = 10
    OPTUNA_FOLDS    = 10
    OPTUNA_EPOCHS   = 10
    FINAL_EPOCHS    = 30
    DEMO_CFG = None

print(f'Device    : {DEVICE}')
print(f'Tasks     : {TASKS}')
print(f'DEMO_MODE : {DEMO_MODE}')

---
## 1 — Model Architecture

Four components in sequence:
1. **Modality encoders** — MLP for M1/M2/M3, GAT for M4, ChemBERTa for M5, SchNet for M6
2. **Protein-conditioned cross-attention** — protein embedding as Query, modality stack as K/V
3. **Modality-Gated MoE fusion** — learned per-target modality weights
4. **Task-specific prediction heads** — one sigmoid head per endpoint

In [ ]:
# ── MLP encoder (for M1 fingerprint, M2 physicochemical, M3 Mordred) ─────────
class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden, n_layers, out_dim, dropout):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout)]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout)]
        layers += [nn.Linear(hidden, out_dim), nn.LayerNorm(out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# ── GAT graph encoder (M4) ────────────────────────────────────────────────────
class GATEncoder(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden, n_layers, heads, out_dim, dropout):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        in_ch = node_dim
        for i in range(n_layers):
            out_ch = hidden // heads if i < n_layers - 1 else hidden
            self.convs.append(GATConv(in_ch, out_ch, heads=heads, edge_dim=edge_dim,
                                      concat=(i < n_layers-1), dropout=dropout))
            self.bns.append(nn.BatchNorm1d(out_ch * heads if i < n_layers-1 else out_ch))
            in_ch = out_ch * heads if i < n_layers-1 else out_ch
        self.proj    = nn.Linear(in_ch, out_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, edge_index, edge_attr, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = F.gelu(bn(conv(x, edge_index, edge_attr=edge_attr)))
            x = self.dropout(x)
        return self.proj(global_mean_pool(x, batch))


# ── ChemBERTa encoder (M5) ────────────────────────────────────────────────────
class ChemBERTaEncoder(nn.Module):
    def __init__(self, model_name, out_dim, dropout):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        self.proj = nn.Sequential(
            nn.Linear(CHEMBERTA_DIM, out_dim), nn.LayerNorm(out_dim), nn.Dropout(dropout))
        for p in self.transformer.parameters():  # freeze by default
            p.requires_grad = False
    def unfreeze_last_n(self, n=2):
        for layer in self.transformer.encoder.layer[-n:]:
            for p in layer.parameters(): p.requires_grad = True
    def forward(self, input_ids, attention_mask):
        out    = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return self.proj(pooled)


# ── SchNet-style 3D encoder (M6) ──────────────────────────────────────────────
class SchNetInteraction(nn.Module):
    def __init__(self, hidden, n_rbf=50):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(n_rbf, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        self.lin = nn.Linear(hidden, hidden)
    def forward(self, h, edge_index, rbf_feat):
        row, col = edge_index
        msg = self.mlp(rbf_feat) * self.lin(h)[col]
        return h + torch.zeros_like(h).scatter_add_(0, row.unsqueeze(-1).expand_as(msg), msg)

class SchNetEncoder(nn.Module):
    def __init__(self, hidden, n_interactions, out_dim, dropout, cutoff=5.0, n_rbf=50):
        super().__init__()
        self.cutoff    = cutoff
        self.atom_emb  = nn.Embedding(119, hidden)
        offsets = torch.linspace(0, cutoff, n_rbf)
        self.register_buffer('offsets', offsets)
        self.coeff     = -0.5 / ((cutoff / n_rbf) ** 2)
        self.interactions = nn.ModuleList(
            [SchNetInteraction(hidden, n_rbf) for _ in range(n_interactions)])
        self.out_proj  = nn.Sequential(
            nn.Linear(hidden, hidden*2), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden*2, out_dim), nn.LayerNorm(out_dim))
    def _rbf(self, dist):
        return torch.exp(self.coeff * (dist.unsqueeze(-1) - self.offsets) ** 2)
    def forward(self, atom_z, pos, batch):
        h = self.atom_emb(atom_z.clamp(0, 118))
        edge_index = radius_graph(pos, r=self.cutoff, batch=batch, loop=False, max_num_neighbors=32)
        if edge_index.size(1) > 0:
            row, col = edge_index
            rbf = self._rbf((pos[row] - pos[col]).norm(dim=-1))
            for interaction in self.interactions:
                h = interaction(h, edge_index, rbf)
        return self.out_proj(global_mean_pool(h, batch))

print('✅ Encoders defined.')

In [ ]:
# ── Protein-Conditioned Cross-Attention ───────────────────────────────────────
class ProteinCrossAttention(nn.Module):
    def __init__(self, dim, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, protein_emb, mod_stack):  # protein: (B,d), mod_stack: (B,M,d)
        Q = protein_emb.unsqueeze(1)             # (B,1,d)
        ctx, weights = self.attn(Q, mod_stack, mod_stack)
        return self.norm(self.drop(ctx.squeeze(1)) + protein_emb), weights


# ── Modality-Gated Mixture-of-Experts Fusion ──────────────────────────────────
class ModalityGatedMoE(nn.Module):
    def __init__(self, dim, n_mod=6, dropout=0.1):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(),
                          nn.Dropout(dropout), nn.Linear(dim*2, dim))
            for _ in range(n_mod)])
        self.gate = nn.Sequential(
            nn.Linear(dim*3, dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim, n_mod))
    def forward(self, h_prot, context, mod_stack):  # mod_stack: (B,M,d)
        gate_in  = torch.cat([h_prot, context, mod_stack.mean(1)], dim=-1)
        gates    = torch.softmax(self.gate(gate_in), dim=-1)  # (B,M)
        expert_out = torch.stack(
            [self.experts[m](mod_stack[:, m, :]) for m in range(len(self.experts))], dim=1)  # (B,M,d)
        fused = (gates.unsqueeze(-1) * expert_out).sum(1)     # (B,d)
        return F.layer_norm(fused + context, fused.shape[-1:]), gates


# ── Full model ────────────────────────────────────────────────────────────────
class MultiModalQSAR(nn.Module):
    def __init__(self, cfg, mord_dim, node_dim=67, edge_dim=10):
        super().__init__()
        d  = cfg['latent_dim']
        h  = cfg['mlp_hidden']
        nl = cfg['mlp_layers']
        dr = cfg['dropout']

        self.fp_enc   = MLPEncoder(2215,    h, nl, d, dr)
        self.phys_enc = MLPEncoder(30,      h, nl, d, dr)   # adjust to actual phys dim
        self.mord_enc = MLPEncoder(mord_dim, h, nl, d, dr)
        self.gat_enc  = GATEncoder(node_dim, edge_dim, h,
                                   cfg['gat_layers'], cfg['gat_heads'], d, dr)
        self.chem_enc = ChemBERTaEncoder(CHEMBERTA_MODEL, d, dr)
        self.sch_enc  = SchNetEncoder(cfg['schnet_hidden'], cfg['schnet_interactions'], d, dr)

        self.prot_proj = nn.Linear(ESM2_DIM, d)
        self.cross_attn = ProteinCrossAttention(d, cfg['attn_heads'], dr)
        self.moe        = ModalityGatedMoE(d, n_mod=6, dropout=dr)

        self.task_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, d//2), nn.GELU(), nn.Dropout(dr), nn.Linear(d//2, 1))
            for _ in range(N_TASKS)])

        # Learnable log-variance for uncertainty weighting (Kendall et al. 2018)
        self.log_vars = nn.Parameter(torch.zeros(N_TASKS))

    def encode_all(self, batch):
        return torch.stack([
            self.fp_enc(batch['fingerprint']),
            self.phys_enc(batch['physicochemical']),
            self.mord_enc(batch['mordred']),
            self.gat_enc(batch['x'], batch['edge_index'], batch['edge_attr'], batch['graph_batch']),
            self.chem_enc(batch['input_ids'], batch['attention_mask']),
            self.sch_enc(batch['atom_z'], batch['coords'], batch['conf_batch']),
        ], dim=1)  # (B, 6, d)

    def forward(self, batch, protein_tensor):
        B = batch['fingerprint'].size(0)
        mod_stack = self.encode_all(batch)    # (B, 6, d)
        preds, gates_all = [], []
        for t in range(N_TASKS):
            h_prot       = self.prot_proj(protein_tensor[t].unsqueeze(0).expand(B, -1))
            context, _   = self.cross_attn(h_prot, mod_stack)
            fused, gates = self.moe(h_prot, context, mod_stack)
            preds.append(self.task_heads[t](fused))
            gates_all.append(gates.detach())
        return torch.cat(preds, dim=1), self.log_vars, gates_all  # (B, N_TASKS)

print('✅ MultiModalQSAR defined.')

---
## 2 — Data Loading

Loads all modalities per task and aligns to a common SMILES intersection.
> Mordred preprocessing (scaler, variance filter) is loaded from nb03 artifacts — not refit.

In [ ]:
# ── Load mordred preprocessors (fit on train in nb03) ────────────────────────
with open(MORD_PREPROC_PATH, 'rb') as f:
    mordred_preprocessors = pickle.load(f)

MORD_DIM = len(mordred_preprocessors[TASKS[0]]['kept_cols'])
print(f'Mordred dim (after preprocessing): {MORD_DIM}')


def load_labels(split):
    dfs = []
    for task in TASKS:
        path = SPLIT_ROOT / task / f'{split}.xlsx'
        if not path.exists(): continue
        df = (pd.read_excel(path)[['Smiles','label']]
              .drop_duplicates('Smiles').rename(columns={'label':task})
              .set_index('Smiles'))
        dfs.append(df)
    return pd.concat(dfs, axis=1) if dfs else pd.DataFrame()

def load_csv_modal(split, modality, skip_cols=('Smiles','pIC50','label','label_name')):
    data = {}
    for task in TASKS:
        path = MOL_REP_ROOT / task / modality / f'{task}_{split}_{modality}.csv'
        if not path.exists(): continue
        df = pd.read_csv(path)
        feat_cols = [c for c in df.columns if c not in skip_cols]
        for _, row in df.iterrows():
            smi = row['Smiles']
            if smi not in data:
                data[smi] = row[feat_cols].values.astype(np.float32)
    return data

def load_pkl_modal(split, modality):
    data = {}
    for task in TASKS:
        path = MOL_REP_ROOT / task / modality / f'{task}_{split}_{modality}.pkl'
        if not path.exists(): continue
        with open(path,'rb') as f: items = pickle.load(f)
        for item in items:
            smi = item['Smiles']
            if smi not in data: data[smi] = item
    return data

def align(label_df, *dicts):
    valid = set(label_df.index)
    for d in dicts: valid &= set(d)
    return sorted([s for s in valid if label_df.loc[s].notna().any()])


print('Loading train pool...')
train_labels = load_labels('train')
train_fp     = load_csv_modal('train', 'fingerprint')
train_phys   = load_csv_modal('train', 'physicochemical')
train_mord   = load_csv_modal('train', 'mordred')   # already standardized in nb03
train_graph  = load_pkl_modal('train', 'graph')
train_conf   = load_pkl_modal('train', 'conformer3d')

train_smiles = align(train_labels, train_fp, train_phys, train_mord, train_graph, train_conf)
print(f'  Aligned train pool: {len(train_smiles)} molecules')

# Task label counts
for task in TASKS:
    if task in train_labels.columns:
        n = train_labels.loc[train_smiles, task].notna().sum()
        print(f'  {task:<22}: {n}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_MODEL)

class MultiModalDataset(Dataset):
    def __init__(self, smiles_list, label_df, fp, phys, mord, graph, conf, tok):
        self.smiles = smiles_list
        self.label_df = label_df
        self.fp = fp; self.phys = phys; self.mord = mord
        self.graph = graph; self.conf = conf; self.tok = tok

    def __len__(self): return len(self.smiles)

    def __getitem__(self, idx):
        smi = self.smiles[idx]
        g   = self.graph[smi]
        c   = self.conf[smi]
        enc = self.tok(smi, return_tensors='pt', padding='max_length',
                       truncation=True, max_length=CHEMBERTA_MAX_LEN)
        row = self.label_df.loc[smi] if smi in self.label_df.index else pd.Series()
        labels, masks = [], []
        for task in TASKS:
            val = row.get(task, np.nan)
            if pd.isna(val): labels.append(0.0); masks.append(0.0)
            else:            labels.append(float(val)); masks.append(1.0)
        return {
            'smiles':          smi,
            'fingerprint':     torch.tensor(self.fp[smi],  dtype=torch.float32),
            'physicochemical': torch.tensor(np.nan_to_num(self.phys[smi], 0.), dtype=torch.float32),
            'mordred':         torch.tensor(self.mord[smi], dtype=torch.float32),
            'x':               torch.tensor(g['node_features'], dtype=torch.float32),
            'edge_index':      torch.tensor(g['edge_index'],    dtype=torch.long).t().contiguous(),
            'edge_attr':       torch.tensor(g['edge_features'], dtype=torch.float32),
            'input_ids':       enc['input_ids'].squeeze(0),
            'attention_mask':  enc['attention_mask'].squeeze(0),
            'atom_z':          torch.tensor(c['atom_types'], dtype=torch.long).clamp(0,118),
            'coords':          torch.tensor(c['coords'],     dtype=torch.float32),
            'labels':          torch.tensor(labels, dtype=torch.float32),
            'masks':           torch.tensor(masks,  dtype=torch.float32),
        }


def collate_fn(batch):
    pyg   = [Data(x=b['x'], edge_index=b['edge_index'], edge_attr=b['edge_attr']) for b in batch]
    gbatch = Batch.from_data_list(pyg)
    az    = [b['atom_z'] for b in batch]
    co    = [b['coords'] for b in batch]
    cb    = torch.cat([torch.full((len(z),), i, dtype=torch.long) for i, z in enumerate(az)])
    return {
        'smiles':          [b['smiles'] for b in batch],
        'fingerprint':     torch.stack([b['fingerprint']     for b in batch]),
        'physicochemical': torch.stack([b['physicochemical'] for b in batch]),
        'mordred':         torch.stack([b['mordred']         for b in batch]),
        'x':               gbatch.x,
        'edge_index':      gbatch.edge_index,
        'edge_attr':       gbatch.edge_attr,
        'graph_batch':     gbatch.batch,
        'input_ids':       torch.stack([b['input_ids']      for b in batch]),
        'attention_mask':  torch.stack([b['attention_mask'] for b in batch]),
        'atom_z':          torch.cat(az),
        'coords':          torch.cat(co),
        'conf_batch':      cb,
        'labels':          torch.stack([b['labels'] for b in batch]),
        'masks':           torch.stack([b['masks']  for b in batch]),
    }

print('✅ Dataset & collate_fn ready.')

---
## 2b — Loss & Metrics

**Loss:** Masked class-weighted BCE with uncertainty weighting (Kendall et al., 2018)  
$$L_{total} = \sum_t \frac{1}{2} e^{-s_t} L_t + \frac{1}{2} s_t$$
where $s_t = \log \sigma_t^2$ is a learnable log-variance per task.

In [ ]:
def masked_uncertainty_loss(preds, labels, masks, log_vars):
    """
    preds    : (B, T) logits
    labels   : (B, T) binary
    masks    : (B, T) 1 if labeled, 0 if missing
    log_vars : (T,)  learnable log-variance per task
    """
    total_loss = 0.
    for t in range(N_TASKS):
        m = masks[:, t].bool()
        if m.sum() == 0: continue
        y_pred = preds[m, t]
        y_true = labels[m, t]
        # class-weighted BCE
        n_pos  = y_true.sum().clamp(min=1)
        n_neg  = (1 - y_true).sum().clamp(min=1)
        w_pos  = (n_pos + n_neg) / (2 * n_pos)
        w_neg  = (n_pos + n_neg) / (2 * n_neg)
        weight = torch.where(y_true == 1, w_pos, w_neg)
        bce    = F.binary_cross_entropy_with_logits(y_pred, y_true, weight=weight)
        # uncertainty weighting
        precision = torch.exp(-log_vars[t])
        total_loss = total_loss + 0.5 * precision * bce + 0.5 * log_vars[t]
    return total_loss


def compute_auroc_per_task(preds_np, labels_np, masks_np):
    aurocs = {}
    for i, task in enumerate(TASKS):
        m  = masks_np[:, i].astype(bool)
        yt = labels_np[m, i]
        yp = torch.sigmoid(torch.tensor(preds_np[m, i])).numpy()
        if len(np.unique(yt)) < 2: continue
        aurocs[task] = roc_auc_score(yt, yp)
    return aurocs

print('✅ Loss & metric functions ready.')

---
## 3 — Optuna Hyperparameter Search (10-Fold Scaffold CV)

> Full run: ~8–24 hours on GPU (10 trials × 10 folds × 10 epochs).  
> Demo run: 2 trials × 1 fold × 5 epochs, ~10 min on CPU.

In [ ]:
# Load protein tensor once
prot_emb_dict  = torch.load(PROT_EMB_PATH, map_location='cpu')
protein_tensor = torch.stack([prot_emb_dict[t] for t in TASKS]).to(DEVICE)  # (N_TASKS, 1280)


def build_config(trial):
    if DEMO_MODE: return DEMO_CFG
    d = trial.suggest_categorical('latent_dim', [256, 512])
    valid_heads = [h for h in [4, 8] if d % h == 0]
    return {
        'latent_dim':          d,
        'mlp_hidden':          trial.suggest_categorical('mlp_hidden', [256, 512]),
        'mlp_layers':          trial.suggest_categorical('mlp_layers', [2, 3]),
        'gat_layers':          trial.suggest_categorical('gat_layers', [2, 3]),
        'gat_heads':           4,
        'schnet_hidden':       trial.suggest_categorical('schnet_hidden', [64, 128]),
        'schnet_interactions': 2,
        'attn_heads':          trial.suggest_categorical('attn_heads', valid_heads),
        'dropout':             trial.suggest_float('dropout', 0.15, 0.35),
        'lr':                  trial.suggest_float('lr', 1e-5, 2e-4, log=True),
        'weight_decay':        trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        'batch_size':          trial.suggest_categorical('batch_size', [8, 16]),
        'unfreeze_epoch':      trial.suggest_categorical('unfreeze_epoch', [10, 999]),
    }


def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.
    for batch in loader:
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            preds, log_vars, _ = model(batch, protein_tensor)
            loss = masked_uncertainty_loss(preds, batch['labels'], batch['masks'], log_vars)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    scheduler.step()
    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_masks = [], [], []
    for batch in loader:
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        preds, _, _ = model(batch, protein_tensor)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(batch['labels'].cpu().numpy())
        all_masks.append(batch['masks'].cpu().numpy())
    return (np.vstack(all_preds), np.vstack(all_labels), np.vstack(all_masks))


def optuna_objective(trial):
    cfg = build_config(trial)
    # Stratify by majority task label
    strat = []
    for s in train_smiles:
        row = train_labels.loc[s] if s in train_labels.index else pd.Series()
        lbl = next((int(row[t]) for t in TASKS if t in row.index and not pd.isna(row.get(t, np.nan))), 0)
        strat.append(lbl)

    skf = StratifiedKFold(n_splits=OPTUNA_FOLDS, shuffle=True, random_state=SEED)
    fold_aurocs = []

    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(train_smiles, strat)):
        tr_smi = [train_smiles[i] for i in tr_idx]
        va_smi = [train_smiles[i] for i in va_idx]

        tr_ds = MultiModalDataset(tr_smi, train_labels, train_fp, train_phys, train_mord,
                                   train_graph, train_conf, tokenizer)
        va_ds = MultiModalDataset(va_smi, train_labels, train_fp, train_phys, train_mord,
                                   train_graph, train_conf, tokenizer)
        tr_dl = DataLoader(tr_ds, batch_size=cfg['batch_size'], shuffle=True,
                           collate_fn=collate_fn, num_workers=0)
        va_dl = DataLoader(va_ds, batch_size=cfg['batch_size'], shuffle=False,
                           collate_fn=collate_fn, num_workers=0)

        model     = MultiModalQSAR(cfg, MORD_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                      weight_decay=cfg['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=OPTUNA_EPOCHS)
        scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))

        best_val_auroc = 0.
        for epoch in range(OPTUNA_EPOCHS):
            # Optionally unfreeze ChemBERTa last 2 layers after N epochs
            if epoch == cfg.get('unfreeze_epoch', 999):
                model.chem_enc.unfreeze_last_n(2)
            train_one_epoch(model, tr_dl, optimizer, scheduler, scaler)
            p, l, m = evaluate(model, va_dl)
            aurocs   = compute_auroc_per_task(p, l, m)
            mean_auc = np.mean(list(aurocs.values())) if aurocs else 0.
            best_val_auroc = max(best_val_auroc, mean_auc)
            if fold_idx == 0:
                trial.report(mean_auc, epoch)
                if trial.should_prune(): raise optuna.exceptions.TrialPruned()

        fold_aurocs.append(best_val_auroc)
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    return float(np.mean(fold_aurocs))


db_path = str(TRAIN_ROOT / 'optuna_10fold.db')
study   = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=2, n_warmup_steps=3),
    study_name='QSAR_10FOLD_CV',
    storage=f'sqlite:///{db_path}',
    load_if_exists=True,
)
completed = len([t for t in study.trials if t.state == TrialState.COMPLETE])
remaining = max(0, N_OPTUNA_TRIALS - completed)
print(f'Completed trials: {completed} | Remaining: {remaining}')

if remaining > 0:
    study.optimize(optuna_objective, n_trials=remaining, show_progress_bar=True,
                   catch=(RuntimeError, torch.cuda.OutOfMemoryError))

best_config = study.best_params if not DEMO_MODE else DEMO_CFG
best_config.setdefault('gat_heads', 4)
best_config.setdefault('schnet_interactions', 2)
print(f'\n✅ Best mean AUROC: {study.best_value:.4f}')
print(json.dumps(best_config, indent=2))

cfg_path = TRAIN_ROOT / 'best_config_10fold.json'
with open(cfg_path, 'w') as f: json.dump(best_config, f, indent=2)
print(f'Config saved → {cfg_path}')

---
## 4 — Final Retrain on Full Train Pool

Using best config from Optuna. Held-out test is **not touched** here.

In [ ]:
full_ds = MultiModalDataset(train_smiles, train_labels, train_fp, train_phys, train_mord,
                             train_graph, train_conf, tokenizer)
full_dl = DataLoader(full_ds, batch_size=best_config['batch_size'], shuffle=True,
                     collate_fn=collate_fn, num_workers=0)

final_model = MultiModalQSAR(best_config, MORD_DIM).to(DEVICE)
optimizer   = torch.optim.AdamW(final_model.parameters(), lr=best_config['lr'],
                                 weight_decay=best_config['weight_decay'])
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINAL_EPOCHS)
scaler      = torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))

print(f'Final retrain: {FINAL_EPOCHS} epochs on {len(full_ds)} molecules')

for epoch in range(FINAL_EPOCHS):
    if epoch == best_config.get('unfreeze_epoch', 999):
        final_model.chem_enc.unfreeze_last_n(2)
    loss = train_one_epoch(final_model, full_dl, optimizer, scheduler, scaler)
    print(f'  Epoch {epoch+1:02d}/{FINAL_EPOCHS} | loss={loss:.4f}')

print('\n✅ Final retrain complete.')

---
## 5 — Save Checkpoint

In [ ]:
ckpt_path = TRAIN_ROOT / 'final_retrain_last.pt'
torch.save({
    'model_state': final_model.state_dict(),
    'config':      best_config,
    'mord_dim':    MORD_DIM,
    'tasks':       TASKS,
    'epoch':       FINAL_EPOCHS,
}, ckpt_path)
print(f'✅ Checkpoint saved → {ckpt_path}')

n_params = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
print(f'   Trainable params: {n_params/1e6:.1f}M')

---
## ✅ Notebook 05 Complete

**Output:**
```
{training_root}/
├── final_retrain_last.pt      ← model weights + config
├── best_config_10fold.json    ← hyperparameters
└── optuna_10fold.db           ← Optuna trial history (SQLite)
```

➡️ **Next:** [06_performance_evaluation.ipynb](06_performance_evaluation.ipynb)